In [ ]:
#해당 파일에서는 데이터 품질/구조 검증을 진행

In [1]:
!pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ------- -------------------------------- 2.6/13.2 MB 12.5 MB/s eta 0:00:01
   ----------- ---------------------------- 3.9/13.2 MB 9.0 MB/s eta 0:00:02
   --------------------- ------------------ 7.1/13.2 MB 11.2 MB/s eta 0:00:01
   ------------------------------- -------- 10.5/13.2 MB 12.3 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 12.5 MB/s eta 0:00:00


In [10]:
# 라이브러리 불러오기
import duckdb

# 월별 원본 Parquet 파일 경로
parquet_path = r"../data/processed/20*.parquet"

In [12]:
'''
event_time	    사용자의 행동이 발생한 시간
event_type	    행동 종류 (view, cart, remove_from_cart, purchase)
product_id	    행동 대상이 된 상품의 고유 ID
category_id	    상품이 속한 카테고리의 고유 ID
category_code	사람이 이해하기 쉬운 상품 카테고리명/분류 (electronics.smartphone 등)
brand	        상품 브랜드명
price	        해당 이벤트 시점의 상품 가격
user_id	        사용자를 구분하는 고유 ID
user_session	한 번의 방문/활동 세션을 구분하는 ID
'''

'\nevent_time\t    사용자의 행동이 발생한 시간\nevent_type\t    행동 종류 (view, cart, remove_from_cart, purchase)\nproduct_id\t    행동 대상이 된 상품의 고유 ID\ncategory_id\t    상품이 속한 카테고리의 고유 ID\ncategory_code\t사람이 이해하기 쉬운 상품 카테고리명/분류 (electronics.smartphone 등)\nbrand\t        상품 브랜드명\nprice\t        해당 이벤트 시점의 상품 가격\nuser_id\t        사용자를 구분하는 고유 ID\nuser_session\t한 번의 방문/활동 세션을 구분하는 ID\n'

In [14]:
# 전체 데이터의 실제 기간과 이벤트 수 확인
duckdb.sql(f"""
    SELECT
        MIN(event_time) AS first_event_time,
        MAX(event_time) AS last_event_time,
        COUNT(*) AS total_events
    FROM read_parquet('{parquet_path}')
""").show()

┌─────────────────────┬─────────────────────┬──────────────┐
│  first_event_time   │   last_event_time   │ total_events │
│      timestamp      │      timestamp      │    int64     │
├─────────────────────┼─────────────────────┼──────────────┤
│ 2019-10-01 00:00:00 │ 2020-04-30 23:59:59 │    411709736 │
└─────────────────────┴─────────────────────┴──────────────┘



In [16]:
# 월별 이벤트 유형별 건수 확인
duckdb.sql(f"""
    SELECT
        STRFTIME(event_time, '%Y-%m') AS month,
        event_type,
        COUNT(*) AS event_count
    FROM read_parquet('{parquet_path}')
    GROUP BY month, event_type
    ORDER BY month, event_type
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬─────────────┐
│  month  │ event_type │ event_count │
│ varchar │  varchar   │    int64    │
├─────────┼────────────┼─────────────┤
│ 2019-10 │ cart       │      926516 │
│ 2019-10 │ purchase   │      742849 │
│ 2019-10 │ view       │    40779399 │
│ 2019-11 │ cart       │     3028930 │
│ 2019-11 │ purchase   │      916939 │
│ 2019-11 │ view       │    63556110 │
│ 2019-12 │ cart       │     3394763 │
│ 2019-12 │ purchase   │     1162048 │
│ 2019-12 │ view       │    62986067 │
│ 2020-01 │ cart       │     2641249 │
│ 2020-01 │ purchase   │      835007 │
│ 2020-01 │ view       │    52490785 │
│ 2020-02 │ cart       │     2885608 │
│ 2020-02 │ purchase   │     1200288 │
│ 2020-02 │ view       │    51232669 │
│ 2020-03 │ cart       │     2968397 │
│ 2020-03 │ purchase   │     1024934 │
│ 2020-03 │ view       │    52347910 │
│ 2020-04 │ cart       │     3268600 │
│ 2020-04 │ purchase   │      966759 │
│ 2020-04 │ view       │    62353909 │
└─────────┴────────────┴─

In [18]:
# 월별 고유 사용자, 세션, 상품 수 확인
duckdb.sql(f"""
    SELECT
        STRFTIME(event_time, '%Y-%m') AS month,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT user_session) AS unique_sessions,
        COUNT(DISTINCT product_id) AS unique_products
    FROM read_parquet('{parquet_path}')
    GROUP BY month
    ORDER BY month
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬──────────────┬─────────────────┬─────────────────┐
│  month  │ unique_users │ unique_sessions │ unique_products │
│ varchar │    int64     │      int64      │      int64      │
├─────────┼──────────────┼─────────────────┼─────────────────┤
│ 2019-10 │      3022290 │         9244421 │          166794 │
│ 2019-11 │      3696117 │        13776050 │          190662 │
│ 2019-12 │      4577232 │        15581360 │          205230 │
│ 2020-01 │      4385985 │        13847854 │          227608 │
│ 2020-02 │      4233206 │        13544595 │          258469 │
│ 2020-03 │      4114060 │        12658462 │          263383 │
│ 2020-04 │      4509623 │        11652261 │          263503 │
└─────────┴──────────────┴─────────────────┴─────────────────┘



In [20]:
# 월별 주요 컬럼 NULL 개수 확인
duckdb.sql(f"""
    SELECT
        STRFTIME(event_time, '%Y-%m') AS month,

        SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS user_id_null,
        SUM(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END) AS session_null,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS product_id_null,
        SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS category_code_null,
        SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) AS brand_null,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS price_null

    FROM read_parquet('{parquet_path}')

    GROUP BY month
    ORDER BY month
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬──────────────┬──────────────┬─────────────────┬────────────────────┬────────────┬────────────┐
│  month  │ user_id_null │ session_null │ product_id_null │ category_code_null │ brand_null │ price_null │
│ varchar │    int128    │    int128    │     int128      │       int128       │   int128   │   int128   │
├─────────┼──────────────┼──────────────┼─────────────────┼────────────────────┼────────────┼────────────┤
│ 2019-10 │            0 │            2 │               0 │           13515609 │    6113008 │          0 │
│ 2019-11 │            0 │           10 │               0 │           21898171 │    9218235 │          0 │
│ 2019-12 │            0 │           21 │               0 │            7088848 │    8115813 │          0 │
│ 2020-01 │            0 │           19 │               0 │            5044890 │    6532738 │          0 │
│ 2020-02 │            0 │           14 │               0 │            4929680 │    8589373 │          0 │
│ 2020-03 │            0 │           

In [22]:
# 일별 View / Cart / Purchase 이벤트 수 계산
duckdb.sql(f"""
    SELECT
        CAST(event_time AS DATE) AS event_date,

        SUM(CASE WHEN event_type = 'view'
                 THEN 1 ELSE 0 END) AS view_count,

        SUM(CASE WHEN event_type = 'cart'
                 THEN 1 ELSE 0 END) AS cart_count,

        SUM(CASE WHEN event_type = 'purchase'
                 THEN 1 ELSE 0 END) AS purchase_count

    FROM read_parquet('{parquet_path}')

    GROUP BY event_date
    ORDER BY event_date
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┬────────────────┐
│ event_date │ view_count │ cart_count │ purchase_count │
│    date    │   int128   │   int128   │     int128     │
├────────────┼────────────┼────────────┼────────────────┤
│ 2019-10-01 │    1208280 │      16658 │          19307 │
│ 2019-10-02 │    1154591 │      17268 │          19469 │
│ 2019-10-03 │    1088725 │      19323 │          19255 │
│ 2019-10-04 │    1346320 │      43829 │          27041 │
│ 2019-10-05 │    1271348 │      35497 │          23494 │
│ 2019-10-06 │    1264062 │      32146 │          22171 │
│ 2019-10-07 │    1161101 │      18052 │          21378 │
│ 2019-10-08 │    1329119 │      18442 │          23072 │
│ 2019-10-09 │    1306363 │      18432 │          22748 │
│ 2019-10-10 │    1243087 │      18997 │          21993 │
│     ·      │       ·    │        ·   │              · │
│     ·      │       ·    │        ·   │              · │
│     ·      │       ·    │        ·   │              · │
│ 2020-04-21 │

In [24]:
# Purchase가 완전히 누락된 날짜 확인
duckdb.sql(f"""
    WITH daily_events AS (
        SELECT
            CAST(event_time AS DATE) AS event_date,

            SUM(CASE WHEN event_type = 'view'
                     THEN 1 ELSE 0 END) AS view_count,

            SUM(CASE WHEN event_type = 'cart'
                     THEN 1 ELSE 0 END) AS cart_count,

            SUM(CASE WHEN event_type = 'purchase'
                     THEN 1 ELSE 0 END) AS purchase_count

        FROM read_parquet('{parquet_path}')

        GROUP BY event_date
    )

    SELECT *
    FROM daily_events

    WHERE purchase_count = 0
    ORDER BY event_date
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┬────────────────┐
│ event_date │ view_count │ cart_count │ purchase_count │
│    date    │   int128   │   int128   │     int128     │
├────────────┼────────────┼────────────┼────────────────┤
│ 2019-11-15 │    5737111 │     483305 │              0 │
│ 2020-01-02 │    1701594 │      71355 │              0 │
└────────────┴────────────┴────────────┴────────────────┘



In [26]:
# 월평균 대비 Purchase가 매우 낮은 날짜 탐색
duckdb.sql(f"""
    WITH daily_purchase AS (
        SELECT
            CAST(event_time AS DATE) AS event_date,
            STRFTIME(event_time, '%Y-%m') AS month,

            SUM(
                CASE
                    WHEN event_type = 'purchase'
                    THEN 1 ELSE 0
                END
            ) AS purchase_count

        FROM read_parquet('{parquet_path}')

        GROUP BY event_date, month
    ),

    monthly_avg AS (
        SELECT
            month,
            AVG(purchase_count) AS avg_purchase_count
        FROM daily_purchase
        GROUP BY month
    )

    SELECT
        d.event_date,
        d.month,
        d.purchase_count,

        ROUND(m.avg_purchase_count, 0)
            AS monthly_avg_purchase,

        ROUND(
            d.purchase_count * 100.0
            / NULLIF(m.avg_purchase_count, 0),
            2
        ) AS pct_of_monthly_avg

    FROM daily_purchase d

    JOIN monthly_avg m
        ON d.month = m.month

    WHERE d.purchase_count < m.avg_purchase_count * 0.2

    ORDER BY d.event_date
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────┬────────────────┬──────────────────────┬────────────────────┐
│ event_date │  month  │ purchase_count │ monthly_avg_purchase │ pct_of_monthly_avg │
│    date    │ varchar │     int128     │        double        │       double       │
├────────────┼─────────┼────────────────┼──────────────────────┼────────────────────┤
│ 2019-11-15 │ 2019-11 │              0 │              30565.0 │                0.0 │
│ 2020-01-01 │ 2020-01 │           3574 │              26936.0 │              13.27 │
│ 2020-01-02 │ 2020-01 │              0 │              26936.0 │                0.0 │
│ 2020-04-20 │ 2020-04 │             22 │              32225.0 │               0.07 │
│ 2020-04-21 │ 2020-04 │             29 │              32225.0 │               0.09 │
└────────────┴─────────┴────────────────┴──────────────────────┴────────────────────┘



In [28]:
# 이상 후보 날짜 전후의 View / Cart / Purchase 비교
duckdb.sql(f"""
    SELECT
        CAST(event_time AS DATE) AS event_date,

        SUM(CASE WHEN event_type = 'view'
                 THEN 1 ELSE 0 END) AS view_count,

        SUM(CASE WHEN event_type = 'cart'
                 THEN 1 ELSE 0 END) AS cart_count,

        SUM(CASE WHEN event_type = 'purchase'
                 THEN 1 ELSE 0 END) AS purchase_count

    FROM read_parquet('{parquet_path}')

    WHERE CAST(event_time AS DATE) BETWEEN '2019-11-13' AND '2019-11-17'
       OR CAST(event_time AS DATE) BETWEEN '2019-12-30' AND '2020-01-04'
       OR CAST(event_time AS DATE) BETWEEN '2020-04-18' AND '2020-04-23'

    GROUP BY event_date
    ORDER BY event_date
""").show()

┌────────────┬────────────┬────────────┬────────────────┐
│ event_date │ view_count │ cart_count │ purchase_count │
│    date    │   int128   │   int128   │     int128     │
├────────────┼────────────┼────────────┼────────────────┤
│ 2019-11-13 │    1924967 │      71650 │          22548 │
│ 2019-11-14 │    2877130 │     170472 │          22124 │
│ 2019-11-15 │    5737111 │     483305 │              0 │
│ 2019-11-16 │    6027932 │     406778 │          68247 │
│ 2019-11-17 │    5783241 │     426941 │         185195 │
│ 2019-12-30 │    2296264 │     148661 │          50729 │
│ 2019-12-31 │    1604276 │     119083 │          38233 │
│ 2020-01-01 │    1428250 │      57620 │           3574 │
│ 2020-01-02 │    1701594 │      71355 │              0 │
│ 2020-01-03 │    1766311 │      95631 │          24377 │
│ 2020-01-04 │    1755349 │      80645 │          28938 │
│ 2020-04-18 │    2416409 │     129944 │          39800 │
│ 2020-04-19 │    2150365 │     114475 │          34089 │
│ 2020-04-20 │

In [30]:
# 의심 구간의 시간대별 로그 누락 여부 확인
duckdb.sql(f"""
    WITH hourly_events AS (
        SELECT
            CAST(event_time AS DATE) AS event_date,
            EXTRACT(HOUR FROM event_time) AS event_hour,

            SUM(CASE WHEN event_type = 'view'
                     THEN 1 ELSE 0 END) AS view_count,

            SUM(CASE WHEN event_type = 'cart'
                     THEN 1 ELSE 0 END) AS cart_count,

            SUM(CASE WHEN event_type = 'purchase'
                     THEN 1 ELSE 0 END) AS purchase_count

        FROM read_parquet('{parquet_path}')

        WHERE CAST(event_time AS DATE)
              BETWEEN '2019-11-14' AND '2019-11-17'

           OR CAST(event_time AS DATE)
              BETWEEN '2020-01-01' AND '2020-01-03'

        GROUP BY
            event_date,
            event_hour
    )

    SELECT *
    FROM hourly_events

    WHERE purchase_count = 0

    ORDER BY
        event_date,
        event_hour
""").show()

┌────────────┬────────────┬────────────┬────────────┬────────────────┐
│ event_date │ event_hour │ view_count │ cart_count │ purchase_count │
│    date    │   int64    │   int128   │   int128   │     int128     │
├────────────┼────────────┼────────────┼────────────┼────────────────┤
│ 2019-11-14 │         19 │     203384 │      20087 │              0 │
│ 2019-11-14 │         20 │     172864 │      14817 │              0 │
│ 2019-11-14 │         21 │     113334 │       8412 │              0 │
│ 2019-11-14 │         22 │      60791 │       4561 │              0 │
│ 2019-11-14 │         23 │      47067 │       3368 │              0 │
│ 2019-11-15 │          0 │      73315 │       6734 │              0 │
│ 2019-11-15 │          1 │     150961 │      13739 │              0 │
│ 2019-11-15 │          2 │     259630 │      22702 │              0 │
│ 2019-11-15 │          3 │     264031 │      27331 │              0 │
│ 2019-11-15 │          4 │     261919 │      28133 │              0 │
│     

In [40]:
# 월별 평소 이벤트 비율과 크게 다른 날짜 탐색
duckdb.sql(f"""
    WITH daily_events AS (
        SELECT
            CAST(event_time AS DATE) AS event_date,
            STRFTIME(event_time, '%Y-%m') AS month,

            SUM(CASE WHEN event_type = 'view'
                     THEN 1 ELSE 0 END) AS view_count,

            SUM(CASE WHEN event_type = 'cart'
                     THEN 1 ELSE 0 END) AS cart_count,

            SUM(CASE WHEN event_type = 'purchase'
                     THEN 1 ELSE 0 END) AS purchase_count

        FROM read_parquet('{parquet_path}')

        GROUP BY event_date, month
    ),

    daily_ratio AS (
        SELECT
            *,
            cart_count * 100.0
                / NULLIF(view_count, 0) AS cart_per_view_pct,

            purchase_count * 100.0
                / NULLIF(view_count, 0) AS purchase_per_view_pct

        FROM daily_events
    ),

    monthly_baseline AS (
        SELECT
            month,

            MEDIAN(cart_per_view_pct)
                AS median_cart_per_view,

            MEDIAN(purchase_per_view_pct)
                AS median_purchase_per_view

        FROM daily_ratio

        GROUP BY month
    )

    SELECT
        d.event_date,
        d.view_count,
        d.cart_count,
        d.purchase_count,

        ROUND(d.cart_per_view_pct, 2)
            AS cart_per_view_pct,

        ROUND(d.purchase_per_view_pct, 2)
            AS purchase_per_view_pct,

        ROUND(m.median_cart_per_view, 2)
            AS monthly_median_cart_per_view,

        ROUND(m.median_purchase_per_view, 2)
            AS monthly_median_purchase_per_view

    FROM daily_ratio d

    JOIN monthly_baseline m
        ON d.month = m.month

    WHERE d.purchase_per_view_pct
              > m.median_purchase_per_view * 3

       OR d.cart_per_view_pct
              < m.median_cart_per_view / 3

    ORDER BY d.event_date
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┬────────────────┬───────────────────┬───────────────────────┬──────────────────────────────┬──────────────────────────────────┐
│ event_date │ view_count │ cart_count │ purchase_count │ cart_per_view_pct │ purchase_per_view_pct │ monthly_median_cart_per_view │ monthly_median_purchase_per_view │
│    date    │   int128   │   int128   │     int128     │      double       │        double         │            double            │              double              │
├────────────┼────────────┼────────────┼────────────────┼───────────────────┼───────────────────────┼──────────────────────────────┼──────────────────────────────────┤
│ 2019-11-01 │    1403991 │      18911 │          22458 │              1.35 │                   1.6 │                         4.42 │                             1.52 │
│ 2019-11-02 │    1514324 │      19350 │          21864 │              1.28 │                  1.44 │                         4.42 │                            

In [32]:
# 2020-02-27 전후 원본 이벤트 수 비교
# 일별 퍼널에서 비정상적으로 낮은 전환율이 관측된 2020-02-27의
# 원본 이벤트 로그를 전후 날짜와 비교하여 로그 누락 여부 확인
duckdb.sql(f"""
    SELECT
        CAST(event_time AS DATE) AS event_date,

        SUM(CASE WHEN event_type = 'view'
                 THEN 1 ELSE 0 END) AS view_count,

        SUM(CASE WHEN event_type = 'cart'
                 THEN 1 ELSE 0 END) AS cart_count,

        SUM(CASE WHEN event_type = 'purchase'
                 THEN 1 ELSE 0 END) AS purchase_count

    FROM read_parquet('{parquet_path}')

    WHERE CAST(event_time AS DATE)
          BETWEEN '2020-02-25' AND '2020-02-29'

    GROUP BY event_date
    ORDER BY event_date
""").show()

┌────────────┬────────────┬────────────┬────────────────┐
│ event_date │ view_count │ cart_count │ purchase_count │
│    date    │   int128   │   int128   │     int128     │
├────────────┼────────────┼────────────┼────────────────┤
│ 2020-02-25 │    1736923 │      83288 │          28063 │
│ 2020-02-26 │    1800712 │      84213 │          27809 │
│ 2020-02-27 │     163441 │       6107 │          27499 │
│ 2020-02-28 │    1003149 │      40604 │          27490 │
│ 2020-02-29 │    1258709 │      48968 │          26316 │
└────────────┴────────────┴────────────┴────────────────┘



In [42]:
# 2019년 11월의 일별 이벤트 구조 전체 확인
duckdb.sql(f"""
    SELECT
        CAST(event_time AS DATE) AS event_date,

        SUM(
            CASE WHEN event_type = 'view'
            THEN 1 ELSE 0 END
        ) AS view_count,

        SUM(
            CASE WHEN event_type = 'cart'
            THEN 1 ELSE 0 END
        ) AS cart_count,

        SUM(
            CASE WHEN event_type = 'purchase'
            THEN 1 ELSE 0 END
        ) AS purchase_count,

        ROUND(
            SUM(CASE WHEN event_type = 'cart'
                     THEN 1 ELSE 0 END) * 100.0
            /
            NULLIF(
                SUM(CASE WHEN event_type = 'view'
                         THEN 1 ELSE 0 END),
                0
            ),
            2
        ) AS cart_per_view_pct,

        ROUND(
            SUM(CASE WHEN event_type = 'purchase'
                     THEN 1 ELSE 0 END) * 100.0
            /
            NULLIF(
                SUM(CASE WHEN event_type = 'view'
                         THEN 1 ELSE 0 END),
                0
            ),
            2
        ) AS purchase_per_view_pct

    FROM read_parquet('{parquet_path}')

    WHERE CAST(event_time AS DATE)
          BETWEEN '2019-11-01' AND '2019-11-30'

    GROUP BY event_date
    ORDER BY event_date
""").show(max_rows=40)

┌────────────┬────────────┬────────────┬────────────────┬───────────────────┬───────────────────────┐
│ event_date │ view_count │ cart_count │ purchase_count │ cart_per_view_pct │ purchase_per_view_pct │
│    date    │   int128   │   int128   │     int128     │      double       │        double         │
├────────────┼────────────┼────────────┼────────────────┼───────────────────┼───────────────────────┤
│ 2019-11-01 │    1403991 │      18911 │          22458 │              1.35 │                   1.6 │
│ 2019-11-02 │    1514324 │      19350 │          21864 │              1.28 │                  1.44 │
│ 2019-11-03 │    1525418 │      20211 │          22145 │              1.32 │                  1.45 │
│ 2019-11-04 │    1744279 │      21960 │          26889 │              1.26 │                  1.54 │
│ 2019-11-05 │    1673138 │      19231 │          24875 │              1.15 │                  1.49 │
│ 2019-11-06 │    1649832 │      19670 │          25319 │              1.19 │     

In [44]:
# 11월 7~9일 시간대별 Cart / View 비율 확인
duckdb.sql(f"""
    SELECT
        CAST(event_time AS DATE) AS event_date,
        EXTRACT(HOUR FROM event_time) AS event_hour,

        SUM(
            CASE WHEN event_type = 'view'
            THEN 1 ELSE 0 END
        ) AS view_count,

        SUM(
            CASE WHEN event_type = 'cart'
            THEN 1 ELSE 0 END
        ) AS cart_count,

        ROUND(
            SUM(CASE WHEN event_type = 'cart'
                     THEN 1 ELSE 0 END) * 100.0
            /
            NULLIF(
                SUM(CASE WHEN event_type = 'view'
                         THEN 1 ELSE 0 END),
                0
            ),
            2
        ) AS cart_per_view_pct

    FROM read_parquet('{parquet_path}')

    WHERE CAST(event_time AS DATE)
          BETWEEN '2019-11-07' AND '2019-11-09'

    GROUP BY
        event_date,
        event_hour

    ORDER BY
        event_date,
        event_hour
""").show(max_rows=100)

┌────────────┬────────────┬────────────┬────────────┬───────────────────┐
│ event_date │ event_hour │ view_count │ cart_count │ cart_per_view_pct │
│    date    │   int64    │   int128   │   int128   │      double       │
├────────────┼────────────┼────────────┼────────────┼───────────────────┤
│ 2019-11-07 │          0 │      12471 │         87 │               0.7 │
│ 2019-11-07 │          1 │      22735 │        174 │              0.77 │
│ 2019-11-07 │          2 │      42661 │        334 │              0.78 │
│ 2019-11-07 │          3 │      59460 │        685 │              1.15 │
│ 2019-11-07 │          4 │      72186 │       1018 │              1.41 │
│ 2019-11-07 │          5 │      77255 │       1134 │              1.47 │
│ 2019-11-07 │          6 │      84075 │       1267 │              1.51 │
│ 2019-11-07 │          7 │      92459 │       1396 │              1.51 │
│ 2019-11-07 │          8 │     108272 │       1535 │              1.42 │
│ 2019-11-07 │          9 │      97769

In [34]:
# 퍼널 분석에서 제외할 로그 이상 날짜
anomaly_dates = [
    "2019-11-14",
    "2019-11-15",
    "2019-11-16",
    "2019-11-17",

    "2020-01-01",
    "2020-01-02",
    "2020-01-03",

    "2020-02-27",

    "2020-04-20",
    "2020-04-21"
]

# SQL NOT IN 절에서 사용할 문자열
anomaly_dates_sql = ", ".join(
    f"'{date}'" for date in anomaly_dates
)

print(anomaly_dates_sql)

'2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17', '2020-01-01', '2020-01-02', '2020-01-03', '2020-02-27', '2020-04-20', '2020-04-21'


In [33]:
# 완전히 동일한 중복 행 개수 확인
duckdb.sql(f"""
    WITH duplicated AS (
        SELECT
            *,
            COUNT(*) AS cnt
        FROM read_parquet('{parquet_path}')
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )

    SELECT
        COUNT(*) AS duplicated_patterns,
        SUM(cnt - 1) AS duplicated_rows
    FROM duplicated
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────┬─────────────────┐
│ duplicated_patterns │ duplicated_rows │
│        int64        │     int128      │
├─────────────────────┼─────────────────┤
│             1001616 │         1384422 │
└─────────────────────┴─────────────────┘



In [37]:
# 완전 중복 행이 어떤 이벤트 유형에서 발생하는지 확인
duckdb.sql(f"""
    WITH duplicated AS (
        SELECT
            event_time,
            event_type,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session,
            COUNT(*) AS cnt

        FROM read_parquet('{parquet_path}')

        GROUP BY
            event_time,
            event_type,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session

        HAVING COUNT(*) > 1
    )

    SELECT
        event_type,
        COUNT(*) AS duplicated_patterns,
        SUM(cnt - 1) AS duplicated_rows

    FROM duplicated

    GROUP BY event_type
    ORDER BY duplicated_rows DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬─────────────────┐
│ event_type │ duplicated_patterns │ duplicated_rows │
│  varchar   │        int64        │     int128      │
├────────────┼─────────────────────┼─────────────────┤
│ cart       │              485821 │          858638 │
│ view       │              364484 │          374449 │
│ purchase   │              151311 │          151335 │
└────────────┴─────────────────────┴─────────────────┘



In [38]:
# 동일한 행이 몇 번씩 반복되는지 확인
duckdb.sql(f"""
    WITH duplicated AS (
        SELECT
            *,
            COUNT(*) AS cnt

        FROM read_parquet('{parquet_path}')

        GROUP BY ALL
        HAVING COUNT(*) > 1
    )

    SELECT
        cnt AS duplicate_count,
        COUNT(*) AS pattern_count

    FROM duplicated

    GROUP BY cnt
    ORDER BY cnt
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬───────────────┐
│ duplicate_count │ pattern_count │
│      int64      │     int64     │
├─────────────────┼───────────────┤
│               2 │        856174 │
│               3 │         73250 │
│               4 │         29008 │
│               5 │         15620 │
│               6 │          8300 │
│               7 │          5327 │
│               8 │          3327 │
│               9 │          2316 │
│              10 │          1741 │
│              11 │          1303 │
│               · │             · │
│               · │             · │
│               · │             · │
│              87 │             2 │
│              88 │             1 │
│              94 │             1 │
│             101 │             1 │
│             106 │             2 │
│             123 │             1 │
│             131 │             2 │
│             132 │             2 │
│             136 │             1 │
│             257 │             1 │
└─────────────────┴─────────

In [41]:
# 월별·이벤트 유형별 완전 중복 행의 규모 확인
duckdb.sql(f"""
    WITH duplicated AS (
        SELECT
            STRFTIME(event_time, '%Y-%m') AS month,
            event_time,
            event_type,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session,
            COUNT(*) AS cnt

        FROM read_parquet('{parquet_path}')

        GROUP BY
            month,
            event_time,
            event_type,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session

        HAVING COUNT(*) > 1
    )

    SELECT
        month,
        event_type,
        SUM(cnt - 1) AS duplicated_rows

    FROM duplicated

    GROUP BY
        month,
        event_type

    ORDER BY
        month,
        event_type
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬─────────────────┐
│  month  │ event_type │ duplicated_rows │
│ varchar │  varchar   │     int128      │
├─────────┼────────────┼─────────────────┤
│ 2019-10 │ cart       │           28073 │
│ 2019-10 │ purchase   │              76 │
│ 2019-10 │ view       │            2071 │
│ 2019-11 │ cart       │           98912 │
│ 2019-11 │ purchase   │               9 │
│ 2019-11 │ view       │            1598 │
│ 2019-12 │ cart       │          110161 │
│ 2019-12 │ purchase   │              20 │
│ 2019-12 │ view       │            1938 │
│ 2020-01 │ cart       │          127259 │
│ 2020-01 │ purchase   │              32 │
│ 2020-01 │ view       │           10305 │
│ 2020-02 │ cart       │          229080 │
│ 2020-02 │ purchase   │          151141 │
│ 2020-02 │ view       │          323695 │
│ 2020-03 │ cart       │          151457 │
│ 2020-03 │ purchase   │              39 │
│ 2020-03 │ view       │           14416 │
│ 2020-04 │ cart       │          113696 │
│ 2020-04 │

In [43]:
# 2020년 2월 purchase 완전 중복의 반복 횟수 분포 확인(다른 월에 비해 2월 purchase가 너무 크기 때문에 추가 확인)
duckdb.sql(f"""
    WITH feb_purchase_duplicates AS (
        SELECT
            event_time,
            event_type,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session,
            COUNT(*) AS cnt

        FROM read_parquet('{parquet_path}')

        WHERE STRFTIME(event_time, '%Y-%m') = '2020-02'
          AND event_type = 'purchase'

        GROUP BY
            event_time,
            event_type,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session

        HAVING COUNT(*) > 1
    )

    SELECT
        cnt AS duplicate_count,
        COUNT(*) AS pattern_count

    FROM feb_purchase_duplicates

    GROUP BY cnt
    ORDER BY cnt
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬───────────────┐
│ duplicate_count │ pattern_count │
│      int64      │     int64     │
├─────────────────┼───────────────┤
│               2 │        151106 │
│               3 │             4 │
│               4 │             9 │
└─────────────────┴───────────────┘



In [45]:
# 2020년 2월 일별 purchase 완전 중복 규모 확인
duckdb.sql(f"""
    WITH feb_purchase_duplicates AS (
        SELECT
            CAST(event_time AS DATE) AS event_date,
            event_time,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session,
            COUNT(*) AS cnt

        FROM read_parquet('{parquet_path}')

        WHERE STRFTIME(event_time, '%Y-%m') = '2020-02'
          AND event_type = 'purchase'

        GROUP BY
            event_date,
            event_time,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session

        HAVING COUNT(*) > 1
    )

    SELECT
        event_date,
        SUM(cnt - 1) AS duplicated_purchase_rows

    FROM feb_purchase_duplicates

    GROUP BY event_date
    ORDER BY event_date
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────────┐
│ event_date │ duplicated_purchase_rows │
│    date    │          int128          │
├────────────┼──────────────────────────┤
│ 2020-02-01 │                        9 │
│ 2020-02-02 │                        3 │
│ 2020-02-03 │                        4 │
│ 2020-02-04 │                        2 │
│ 2020-02-05 │                        3 │
│ 2020-02-06 │                        3 │
│ 2020-02-07 │                        1 │
│ 2020-02-08 │                        3 │
│ 2020-02-10 │                    10135 │
│ 2020-02-11 │                    15362 │
│     ·      │                      ·   │
│     ·      │                      ·   │
│     ·      │                      ·   │
│ 2020-02-17 │                    16171 │
│ 2020-02-18 │                     2607 │
│ 2020-02-19 │                        2 │
│ 2020-02-20 │                        1 │
│ 2020-02-21 │                        1 │
│ 2020-02-22 │                        1 │
│ 2020-02-23 │                    

In [47]:
# 2020년 2월에서 purchase 중복이 크게 발생한 날짜만 확인
duckdb.sql(f"""
    WITH feb_purchase_duplicates AS (
        SELECT
            CAST(event_time AS DATE) AS event_date,
            event_time,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session,
            COUNT(*) AS cnt

        FROM read_parquet('{parquet_path}')

        WHERE STRFTIME(event_time, '%Y-%m') = '2020-02'
          AND event_type = 'purchase'

        GROUP BY
            event_date,
            event_time,
            product_id,
            category_id,
            category_code,
            brand,
            price,
            user_id,
            user_session

        HAVING COUNT(*) > 1
    ),

    daily_duplicates AS (
        SELECT
            event_date,
            SUM(cnt - 1) AS duplicated_purchase_rows

        FROM feb_purchase_duplicates

        GROUP BY event_date
    )

    SELECT *
    FROM daily_duplicates

    WHERE duplicated_purchase_rows >= 100

    ORDER BY event_date
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────────┐
│ event_date │ duplicated_purchase_rows │
│    date    │          int128          │
├────────────┼──────────────────────────┤
│ 2020-02-10 │                    10135 │
│ 2020-02-11 │                    15362 │
│ 2020-02-12 │                    25368 │
│ 2020-02-13 │                    21372 │
│ 2020-02-14 │                    19450 │
│ 2020-02-15 │                    18174 │
│ 2020-02-16 │                    22453 │
│ 2020-02-17 │                    16171 │
│ 2020-02-18 │                     2607 │
└────────────┴──────────────────────────┘



In [46]:
## 데이터 품질 검증 결론

전체 데이터는 2019-10-01부터 2020-04-30까지 존재하지만,
2019-11-08 전후로 Cart/View 비율이 불연속적으로 변화하는 패턴이 확인되었다.

특히 2019-11-07까지는 Cart/View 비율이 대체로 1%대였으나,
2019-11-08 새벽부터 3~6% 수준으로 급격히 변화하였다.

현재 데이터만으로 정확한 원인을 확인할 수 없으므로
2019년 10~11월 데이터를 이후 기간과 동일한 측정 기준으로 비교하는 것은
위험하다고 판단하였다.

따라서 월별 구매 퍼널 비교의 메인 분석 기간은
측정 구조가 상대적으로 안정적인 2019-12-01 ~ 2020-04-30으로 제한한다.

단, 2019년 10~11월 데이터 자체를 삭제하지는 않으며
전체 데이터 품질 검증 및 초기 관측 구간의 참고 자료로 유지한다.

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (1096405318.py, line 3)

In [48]:
### 메인 퍼널 분석에서 추가 제외하는 로그 이상 날짜

- 2020-01-01 ~ 2020-01-03: Purchase 로그 부분/전체 누락
- 2020-02-27: View/Cart 로그 대량 누락
- 2020-04-20 ~ 2020-04-21: Purchase 로그 거의 전부 누락

또한 2020-02-10 ~ 2020-02-18에는 동일 Purchase 행의 중복 기록이
집중적으로 확인되었으며, Purchase 횟수를 직접 사용하는 분석에서는
중복 영향을 별도로 통제한다.

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (1196558788.py, line 3)